# Imports

In [9]:
import math
import numpy as np
import matplotlib.pyplot as plt
import graphviz

%matplotlib inline

# Utils

# Exercise

We just implemented the backward function for the `__add__`. 


**Goal**: Modify this class to implement the `_backward()` functionality for the remaining operations: subtraction, multiplication, division, and tanh.

In [10]:
class Value:

    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self.grad = 0.0 # at initialization every Value does not impact the output
        self._backward = lambda: None # this method does the chain rule and stores how it transmits the output's gradient into the inputs' gradient of the current node
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value({self.data})"
    
    def __add__(self, other: "Value"):
        out =  Value(self.data + other.data, (self,other), '+')

        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out
    
    def __mul__(self, other: "Value"):
        out =  Value(self.data * other.data, (self,other), '*')
        return out

    def __sub__(self, other: "Value"):
        out = Value(self.data - other.data, (self,other), '-')
        return out
    
    def __truediv__(self, other: "Value"):
        out = Value(self.data / other.data, (self,other), '/')
        return out
    
    def tanh(self):
        x = self.data
        t = (math.exp(2*x)-1) / (math.exp(2*x) + 1)
        out = Value(t, (self, ), 'tanh')
        return out 

test

In [86]:
a = Value(2.0, label='a')
b = Value(3.0, label='b')

c = a + b
# check that the gradient in a is 1
c.grad = 1.0
c._backward()

assert a.grad == 1.0, "gradient through an addition is wrong"
assert b.grad == 1.0, "gradient through an addition is wrong"

# now let's check the gradient through multiplication
a = Value(2.0, label='a')
b = Value(3.0, label='b')

c = a * b
c.grad = 3.0
c._backward()

assert a.grad == 9, "gradient through multiplication is wrong"
assert b.grad == 6, "gradient through multiplication is wrong"

# now let's check the gradient through subtraction
a = Value(2.0, label='a')
b = Value(3.0, label='b')

c = a - b
c.grad = 4.0
c._backward()

assert a.grad == 4.0, "gradient through subtraction is wrong"
assert b.grad == -4.0, "gradient through subtraction is wrong"

# now let's check the gradient through division
a = Value(2.0, label='a')
b = Value(3.0, label='b')

c = a / b
c.grad = 5.0
c._backward()

assert np.allclose(a.grad, 5.0/3.0), "gradient through division is wrong"
assert np.allclose(b.grad, -10/9), "gradient through division is wrong"

# now let's check the gradient through the tanh function
a = Value(0.549306144, label='a')
c = a.tanh()
c.grad = 2.0
c._backward()

assert np.allclose(a.grad,1.5), "gradient through tanh is wrong"